## This is the code to train the model and acquire influence for Neighbour Influence 1 Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. What you expect here is to acquire the estimation result for a certrain experiment setting. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **For Neighbour Influence Experiment, you don't need to run several times. But for multi-verification, you can still do the following steps** -> Change the Training Sample Size and Repeat all the process -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [34]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [35]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [36]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [37]:
import random
from keras.optimizers import SGD

In [38]:
from sklearn.datasets import make_classification

In [39]:
import time

Train_Size: 1000, 2000, 4000, 8000, 16000  
Feature_Size: 10, 20, 40, 80, 160

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The only thing you might need to change in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training samples might be changed to do multi-verification experiments.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [40]:
train_pool = 16000
test_size = 500
train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
n_features=160
seed=42
sep = 5

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [41]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [42]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      -0.030731   5.104282   1.904383   3.000913  -2.631929  -0.303317   
1       9.674122  -6.042198  -0.361050  -4.194802  -3.174503  -6.086205   
2       3.554483   4.024516   0.718647   9.142869  -7.724641 -17.115361   
3      10.247322   9.297177  19.781251  -5.975640  -0.112818  -4.143577   
4       5.366878   8.723623   2.859152 -11.458412   2.772342   6.690669   
...          ...        ...        ...        ...        ...        ...   
16495  11.789845   2.118917   0.086399  -7.917333   8.914663   1.216330   
16496  -8.512763   3.450824   8.676382   3.684412   3.430197   2.687541   
16497  -6.435202  12.781332  -7.879941   2.550949  10.517249  -5.200181   
16498  -1.193569   8.441643  -0.230270  -5.256229  -9.912854  -5.637027   
16499 -17.112508  -4.698404 -10.014821 -16.395224  -6.951845 -13.319951   

       feature_7  feature_8  feature_9  feature_10  ...  feature_153  \
0      11.699076  10.243957

In [43]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [44]:
print(df_train_pool.head())
print(df_test.head())

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0  -0.030731   5.104282   1.904383   3.000913  -2.631929  -0.303317   
1   9.674122  -6.042198  -0.361050  -4.194802  -3.174503  -6.086205   
2   3.554483   4.024516   0.718647   9.142869  -7.724641 -17.115361   
3  10.247322   9.297177  19.781251  -5.975640  -0.112818  -4.143577   
4   5.366878   8.723623   2.859152 -11.458412   2.772342   6.690669   

   feature_7  feature_8  feature_9  feature_10  ...  feature_153  feature_154  \
0  11.699076  10.243957  -8.505072    8.585870  ...    -6.000722    -0.260021   
1   8.550103  15.591247   4.462937    3.210544  ...    -5.223419    -3.699902   
2  -8.333522   7.730190   6.349771   13.636357  ...    10.183081    -5.300087   
3  15.289449  -3.964560  -1.731965   16.514199  ...    16.744368    -3.023920   
4  21.290693  14.850827  12.231748   16.023459  ...    17.033173   -11.559994   

   feature_155  feature_156  feature_157  feature_158  feature_159  \
0     3.394554  

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

In [45]:
features_to_test = 10
selected_features = [f'feature_{i+1}' for i in range(features_to_test)]

nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]
nested_train_dfs = [df[ selected_features + ['label', 'id'] ].copy()for df in nested_train_dfs]

df_test = df_test[ selected_features + ['label', 'id'] ].copy()

core_1000_ids = nested_train_dfs[0]['id'].tolist()

In [46]:
train_df = nested_train_dfs[7]

In [47]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[-3.0731456e-02  5.1042824e+00  1.9043831e+00 ... -8.5050716e+00
   8.5858698e+00  1.0000000e-10]
 [ 9.6741228e+00 -6.0421977e+00 -3.6105025e-01 ...  4.4629364e+00
   3.2105439e+00  2.0000000e-10]
 [ 3.5544834e+00  4.0245156e+00  7.1864682e-01 ...  6.3497705e+00
   1.3636357e+01  3.0000000e-10]
 ...
 [-8.5129566e+00 -2.5630813e+00  9.5491819e+00 ... -4.5513077e+00
  -1.0173520e+01  7.9979998e-07]
 [-7.3252635e+00  1.1118751e+01  8.5200033e+00 ...  9.6676655e+00
   9.4154425e+00  7.9990002e-07]
 [-1.5232234e+00  1.8553445e+00 -7.1496618e-01 ...  1.1264756e+01
  -5.3503485e+00  8.0000001e-07]]


In [48]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[-4.1068168e+00  1.6207395e+00  2.8891449e+00 ...  1.6416637e+01
  -5.2383075e+00  1.6000999e-06]
 [ 1.7714390e+01 -6.0222381e-01  1.4438127e+01 ...  1.6587117e+00
   4.0881343e+00  1.6002000e-06]
 [ 9.1585236e+00 -8.9030685e+00  3.0126219e+00 ...  3.4371374e+00
   6.9341512e+00  1.6003000e-06]
 ...
 [-6.4352021e+00  1.2781332e+01 -7.8799405e+00 ...  2.1908827e+00
   2.1765578e+00  1.6498000e-06]
 [-1.1935693e+00  8.4416428e+00 -2.3027022e-01 ...  3.9752128e+00
  -1.4689459e+01  1.6499000e-06]
 [-1.7112509e+01 -4.6984038e+00 -1.0014821e+01 ...  9.5027094e+00
  -7.4139982e-01  1.6499999e-06]]


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [49]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [50]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS
# import seaborn as sns
# import matplotlib.pyplot as plt

In [51]:
# D = pairwise_distances(X_all) 

In [52]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [53]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [54]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [55]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [56]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.7834 - accuracy: 0.5234 - val_loss: 0.7032 - val_accuracy: 0.5480 - 541ms/epoch - 17ms/step
32/32 - 0s - loss: 0.5659 - accuracy: 0.7131 - val_loss: 0.5423 - val_accuracy: 0.7680 - 78ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4751 - accuracy: 0.8081 - val_loss: 0.4746 - val_accuracy: 0.8220 - 65ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4319 - accuracy: 0.8281 - val_loss: 0.4366 - val_accuracy: 0.8360 - 61ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4044 - accuracy: 0.8413 - val_loss: 0.4104 - val_accuracy: 0.8460 - 63ms/epoch - 2ms/step
32/32 - 0s - loss: 0.3838 - accuracy: 0.8511 - val_loss: 0.3903 - val_accuracy: 0.8520 - 61ms/epoch - 2ms/step
32/32 - 0s - loss: 0.3673 - accuracy: 0.8575 - val_loss: 0.3744 - val_accuracy: 0.8560 - 67ms/epoch - 2ms/step
32/32 - 0s - loss: 0.3536 - accuracy: 0.8633 - val_loss: 0.3614 - val_accuracy: 0.8640 - 92ms/epoch - 3ms/step
32/32 - 0s - loss: 0.3421 - accuracy: 0.8665 - val_loss: 0.3507 - val_accuracy: 0.8640 - 101ms/epoch - 3ms/ste

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [57]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [58]:
start_if = time.perf_counter()

In [60]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0            1  0.016705
1            2  0.007911
2            3  0.025332
3            4  0.004754
4            5  0.001116
...        ...       ...
7995      7996  0.012413
7996      7997  0.029897
7997      7998  0.037342
7998      7999  0.016962
7999      8000  0.051272

[8000 rows x 2 columns]


In [61]:
end_if = time.perf_counter()
runtime_if = end_if - start_if
print(f"FOIF Runtime: {runtime_if:.4f} seconds")

FOIF Runtime: 207.8934 seconds


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [62]:
start_tc = time.perf_counter()

In [63]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0            1 -0.000125
1            2 -0.000054
2            3 -0.000210
3            4 -0.000013
4            5 -0.000002
...        ...       ...
7995      7996  0.000462
7996      7997 -0.000207
7997      7998  0.002015
7998      7999 -0.001810
7999      8000  0.002627

[8000 rows x 2 columns]


In [64]:
end_tc = time.perf_counter()
runtime_tc = end_tc - start_tc
print(f"TracIn Runtime: {runtime_tc:.4f} seconds")

TracIn Runtime: 345.8695 seconds


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [65]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [66]:
TracIn_sorted.to_csv("TC_Train_Set_8_seed42.csv",index = False)
df_sorted.to_csv("IF_Train_Set_8_seed42.csv",index = False)